In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gallo33henrique/chest-x-ray-and-oct-medical-image-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-x-ray-and-oct-medical-image-dataset' dataset.
Path to dataset files: /kaggle/input/chest-x-ray-and-oct-medical-image-dataset


In [2]:
!ls /kaggle/input/chest-x-ray-and-oct-medical-image-dataset/data

test  train


In [3]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras import optimizers ,layers
from sklearn.preprocessing import LabelEncoder


In [4]:
# تحويل الصور لـ grayscale
def load_dataset_grayscale(directory):
    images = []
    labels = []

    for label in os.listdir(directory):
        label_dir = os.path.join(directory, label)
        if not os.path.isdir(label_dir):
            continue


        for image_file in os.listdir(label_dir):
            image_path = os.path.join(label_dir, image_file)

            # قراءة كـ grayscale مباشرة
            image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue

            image = cv2.resize(image, (128, 128))
            images.append(image)
            labels.append(label)

    # إضافة dimension للـ channel
    images = np.array(images)
    images = np.expand_dims(images, axis=-1)  # Shape: (n, 128, 128, 1)


    return images, np.array(labels)
data_dir = os.path.join(path, 'data')
train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'test')

# إعادة تحميل الداتا بـ grayscale
x_train_gray, y_train = load_dataset_grayscale(train_dir)
x_test_gray, y_test = load_dataset_grayscale(test_dir)

# Split validation
from sklearn.model_selection import train_test_split
x_train_gray, x_val_gray, y_train, y_val = train_test_split(
    x_train_gray, y_train, test_size=0.2, stratify=y_train, random_state=42
)

# Normalize
x_train = x_train_gray.astype('float32') / 255.0
x_val = x_val_gray.astype('float32') / 255.0
x_test = x_test_gray.astype('float32') / 255.0

# Encode labels
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_val = label_encoder.transform(y_val)
y_test = label_encoder.transform(y_test)

In [5]:
x_train.shape

(4185, 128, 128, 1)

In [6]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [7]:
# بناء المودل
model = Sequential([
    # layer 1
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(128,128,1)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # layer 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # layer 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    # Dense layers
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

# ملخص المودل
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     8,388,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,562,657 (32.66 MB)

 Trainable params: 8,561,697 (32.66 MB)

 Non-trainable params: 960 (3.75 KB)

In [ ]:
# Callbacks للتدريب
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
]

# التدريب
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 500s 4s/step - accuracy: 0.8820 - loss: 0.2673 - precision: 0.9432 - recall: 0.8982 - val_accuracy: 0.7421 - val_loss: 32.7497 - val_precision: 0.7421 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 501s 4s/step - accuracy: 0.9431 - loss: 0.1602 - precision: 0.9595 - recall: 0.9634 - val_accuracy: 0.7421 - val_loss: 7.1253 - val_precision: 0.7421 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 3/10
122/131 ━━━━━━━━━━━━━━━━━━━━ 31s 4s/step - accuracy: 0.9602 - loss: 0.1092 - precision: 0.9747 - recall: 0.9712

In [ ]:
# التقييم
test_loss, test_acc, test_prec, test_rec = model.evaluate(x_test, y_test)
print(f"\n Result Test Set:")
print(f"Accuracy: {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall: {test_rec:.4f}")

In [ ]:
# حفظ المودل
model.save('chest_xray_model.h5')
print("\n Model Saved : chest_xray_model.h5")


In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'],label = 'Training')
plt.plot(history.history['val_accuracy'],label = 'Validation')
plt.title('Training vs Validation Accuracy ')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show